# Fig. 6: SAMap transfer of mouse cell type and pseudotime onto the human nuclei

Reads the SAMap object (`SAMAP_FINAL`, the deposited `Samap_Final.h5ad`) and the human fibroblast/decidual object
(`HUMAN_H5AD`, for the section positions) and writes panels 6A, 6B, 6C, 6E and 6D into `OUT` as SVG, with a color key
for each section pair and a provenance file per panel. `OUT` must be a new folder outside `xmens_figures`; copy checked
panels into `xmens_figures/final_figures/Fig6_crossspecies_radial_patterning/` by hand.

What each panel shows:

- 6A: the joint SAMap UMAP, every nucleus colored by species.
- 6B: the joint SAMap UMAP, human nuclei colored by `obs['transferred_mouse_fibdec']`, mouse nuclei grey underneath.
  The vote is recomputed from `obsp['connectivities']` as a check and must agree with the stored column.
- 6C: the same UMAP, human nuclei colored by `obs['transferred_mouse_pseudotime']`.
- 6D: transferred cell type on the same sections.
- 6E: transferred pseudotime on the secretory and menstrual sections.

Drawing rules: one viridis scale from 0 to 1 for every pseudotime panel, values as stored; every nucleus drawn;
nuclei without a transferred value in grey underneath. Positions are `x_um_dbscan`, `y_um_dbscan` of the human
object. Font: Helvetica Neue if `FONT_FILE` points to it, otherwise DejaVu Sans.

The drawing code of 6B, 6C and 6E is that of `samap/notebooks/5_figures/` (27 Oct 2026).

In [ ]:
# Parameters (papermill -p NAME VALUE overrides them)
OUT = ""  # a new folder for this figure's panels
SAMAP_FINAL = "/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_11_xmens_samap_final/3_transfer/Samap_Final.h5ad"
HUMAN_H5AD = "/n/lab_storage/eddy_lab/Lab/mckinley/data/20261107_Human_fibdec_final_nick_pdt.h5ad"  # positions
FONT_FILE = ""  # a Helvetica Neue font file (optional); empty: DejaVu Sans
DATE = "20261111"  # file-name prefix of the section panels

In [ ]:
import datetime
import hashlib
import json
import platform
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from IPython.display import SVG, display
from matplotlib import font_manager
from matplotlib.lines import Line2D

OUT = Path(OUT)
assert OUT.name, "set OUT, the folder for this figure's panels"
assert "xmens_figures" not in OUT.resolve().parts, "write to OUT; copy checked panels into xmens_figures by hand"
assert not OUT.exists(), f"{OUT} exists; use a new folder"
OUT.mkdir(parents=True)

SAMAP_FINAL, HUMAN_H5AD = Path(SAMAP_FINAL), Path(HUMAN_H5AD)
TRANSFERRED = "transferred_mouse_pseudotime"  # the transfer, in the SAMap object's obs
TEXT = "white"


def set_up_font(font_file: str) -> str:
    """Helvetica Neue from a font file if given, otherwise DejaVu Sans; returns the family in use."""
    family = ["DejaVu Sans"]
    if font_file:
        font_manager.fontManager.addfont(font_file)
        family = ["Helvetica Neue", "DejaVu Sans"]
    mpl.rcParams.update({"font.family": family, "svg.fonttype": "none", "pdf.fonttype": 42})
    return font_manager.FontProperties(family=family).get_name()


def file_record(path: Path, role: str) -> dict:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return {"path": str(path.resolve()), "role": role, "bytes": path.stat().st_size, "sha256": digest.hexdigest()}


def provenance(name: str, outputs: list, inputs: list, **extra) -> None:
    """A JSON record beside the panel: inputs with checksums, outputs, settings, versions."""
    record = {"panel": name, "written": datetime.datetime.now().isoformat(timespec="seconds"),
              "host": platform.node(), "notebook": "figures/Fig6_crossspecies_radial_patterning.ipynb",
              "samap_final": str(SAMAP_FINAL), "inputs": inputs, "outputs": [str(p.resolve()) for p in outputs],
              "font": font, "versions": {"python": platform.python_version(), "matplotlib": mpl.__version__,
                                         "pandas": pd.__version__}, **extra}
    (OUT / f"{name}.provenance.json").write_text(json.dumps(record, indent=1, default=str))


font = set_up_font(FONT_FILE)

# The SAMap object, backed: obs, the joint UMAP and the graph are read; X stays on disk.
samap = ad.read_h5ad(SAMAP_FINAL, backed="r")
joint = samap.obs.astype({c: object for c in ("species", "sample", "timepoint", "cell_type",
                                              "transferred_mouse_fibdec")})  # plain strings, NaN where unmapped
joint.index.name = "cell"
joint[["X_umap__0", "X_umap__1"]] = np.asarray(samap.obsm["X_umap"])

# Section positions of the human nuclei, from the human object.
positions = ad.read_h5ad(HUMAN_H5AD, backed="r").obs[["x_um_dbscan", "y_um_dbscan"]]
positions.columns = ["x_um", "y_um"]
positions.index.name = "cell"

human = joint[joint.species == "hs"].join(positions, how="inner")
assert len(human) == (joint.species == "hs").sum(), "human nuclei of the SAMap object missing from the human object"
mouse = joint[joint.species == "mm"]
inputs = [file_record(SAMAP_FINAL, "samap_final"), file_record(HUMAN_H5AD, "human_object")]
print(f"font: {font}")
print(f"{len(human):,} human and {len(mouse):,} mouse nuclei; {int(human[TRANSFERRED].notna().sum()):,} human "
      f"nuclei with a transferred pseudotime; {int(human.x_um.notna().sum()):,} human nuclei positioned")


## The joint UMAP panels (6B, 6C)

90 x 90 mm on black, one square frame fitted to the whole joint embedding so the panels overlay each other, two short
axis arrows instead of a box, 600 dpi. Mouse nuclei underneath in dark grey.

In [ ]:
CMAP = mpl.colormaps["viridis"]
NORM = mpl.colors.Normalize(0, 1)
MOUSE_CONTEXT = "#3a3a3a"  # mouse nuclei: their shape shows, their color does not compete
NO_VALUE_UMAP = "#9a9a9a"  # human nuclei without a transferred value
AXIS_ARROWS = "#bbbbbb"
PANEL_MM = 90
UMAP_DOT = 0.5            # pt^2
NO_VALUE_DOT_SCALE = 3    # larger, so the few nuclei without a value stay visible
UMAP_DPI = 600
ORDER_SEED = 3            # seeded random draw order, so neither early nor late nuclei always sit on top
CELL_TYPE_COLORS = {"endometrial_fibroblast": "#CC79A7", "decidual_cell": "#00B4C5"}
CELL_TYPE_LABELS = {"endometrial_fibroblast": "eFib", "decidual_cell": "dec"}


def square_limits(umap: pd.DataFrame) -> tuple:
    """Square frame around the full joint embedding (both species), 6% wider than its larger extent."""
    x, y = umap["X_umap__0"], umap["X_umap__1"]
    side = max(x.max() - x.min(), y.max() - y.min()) * 1.06
    cx, cy = (x.max() + x.min()) / 2, (y.max() + y.min()) / 2
    return (cx - side / 2, cx + side / 2), (cy - side / 2, cy + side / 2), side


def umap_figure(title: str, xlim, ylim, side):
    """Black square panel with a boxless equal-aspect UMAP axes and short axis arrows."""
    fig = plt.figure(figsize=(PANEL_MM / 25.4, PANEL_MM / 25.4), facecolor="black")
    ax = fig.add_axes([0.06, 0.06, 0.88, 0.84])
    ax.set_facecolor("black")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.text(0.06, 0.965, title, ha="left", va="top", fontsize=9, color=TEXT)

    # UMAP coordinates have no units, so two short arrows replace a box and ticks.
    length = 0.14 * side
    arrow = dict(arrowprops=dict(arrowstyle="-|>", lw=0.8, color=AXIS_ARROWS, mutation_scale=6, shrinkA=0,
                                 shrinkB=0),
                 annotation_clip=False)
    ax.annotate("", xy=(xlim[0] + length, ylim[0]), xytext=(xlim[0], ylim[0]), **arrow)
    ax.annotate("", xy=(xlim[0], ylim[0] + length), xytext=(xlim[0], ylim[0]), **arrow)
    ax.text(xlim[0] + 0.015 * side, ylim[0] - 0.012 * side, "UMAP 1", ha="left", va="top", fontsize=7,
            color=AXIS_ARROWS)
    ax.text(xlim[0] - 0.015 * side, ylim[0] + 0.015 * side, "UMAP 2", ha="right", va="bottom", rotation=90,
            fontsize=7, color=AXIS_ARROWS)
    return fig, ax


def scatter_umap(ax, cells: pd.DataFrame, colors, zorder: int, size: float = UMAP_DOT) -> None:
    ax.scatter(cells["X_umap__0"], cells["X_umap__1"], s=size, c=colors, linewidths=0, rasterized=True,
               zorder=zorder)


def colorbar_vertical(fig) -> None:
    cax = fig.add_axes([0.80, 0.66, 0.025, 0.20])
    bar = fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=cax, ticks=[0, 0.5, 1])
    bar.ax.set_yticklabels(["0 early", "0.5", "1 late"], fontsize=7, color=TEXT)
    bar.outline.set_visible(False)
    bar.ax.tick_params(length=1.5, pad=1.5, color=TEXT)
    cax.set_title("pseudotime\n(rank)", fontsize=7, loc="left", pad=4, color=TEXT)


def key(fig, rect, entries, title=None) -> None:
    """Dot key in its own axes: cell types top right (where 6C has its color bar); grey nuclei bottom right."""
    handles = [Line2D([], [], ls="", marker="o", ms=4, mfc=color, mec="none", label=label)
               for color, label in entries]
    kax = fig.add_axes(rect)
    kax.axis("off")
    legend = kax.legend(handles=handles, loc="upper left" if title else "lower left", fontsize=7,
                        handletextpad=0.3, labelspacing=0.4, borderaxespad=0, labelcolor=TEXT, frameon=False,
                        title=title, title_fontsize=7, alignment="left")
    if title:
        legend.get_title().set_color(TEXT)


def write_umap_panels(stem: str, title: str, colored: pd.DataFrame, colors, grey: pd.DataFrame, grey_label: str,
                      key_top_right=None) -> list:
    """One UMAP panel: mouse nuclei in dark grey underneath, human nuclei colored; returns the files written."""
    xlim, ylim, side = square_limits(joint)
    fig, ax = umap_figure(title, xlim, ylim, side)
    scatter_umap(ax, mouse, MOUSE_CONTEXT, zorder=1)
    # Human nuclei without a value sit behind the colored ones, above the mouse context.
    scatter_umap(ax, grey, NO_VALUE_UMAP, zorder=2, size=UMAP_DOT * NO_VALUE_DOT_SCALE)
    scatter_umap(ax, colored, colors, zorder=3)
    if key_top_right is None:
        colorbar_vertical(fig)
    else:
        key(fig, [0.80, 0.66, 0.20, 0.22], key_top_right, title="transferred\nfrom mouse")
    key(fig, [0.56, 0.0, 0.44, 0.16], [(MOUSE_CONTEXT, "mouse cells"), (NO_VALUE_UMAP, grey_label)])
    path = OUT / f"{stem}.svg"
    fig.savefig(path, dpi=UMAP_DPI, facecolor="black")
    plt.close(fig)
    return [path]


## 6A: the joint UMAP by species

Every nucleus of the SAMap object, mouse in orange and human in blue, in a seeded random draw order so neither
species sits on top. Same square, dot size and axis arrows as 6B and 6C, so the three panels overlay. Colors and
title are those of the record (`xmens_final` `joint_umap_panel.py`).

In [ ]:
SPECIES_COLORS = {"mm": "#E6782A", "hs": "#2A6FDB"}
SPECIES_LABELS = {"mm": "mouse", "hs": "human"}

cells_6a = joint.sample(frac=1, random_state=ORDER_SEED)
xlim, ylim, side = square_limits(joint)
fig, ax = umap_figure("Human and mouse cells co-embedded by SAMap", xlim, ylim, side)
scatter_umap(ax, cells_6a, cells_6a.species.map(SPECIES_COLORS).to_list(), zorder=1)
key(fig, [0.56, 0.0, 0.44, 0.16], [(SPECIES_COLORS[s], SPECIES_LABELS[s]) for s in ("mm", "hs")])
path = OUT / "Fig6A_umap_species.svg"
fig.savefig(path, dpi=UMAP_DPI, facecolor="black")
files_6a = [path]
plt.close(fig)
counts_6a = {SPECIES_LABELS[s]: int((joint.species == s).sum()) for s in ("mm", "hs")}
print("6A:", counts_6a)
provenance("Fig6A_umap_species", files_6a, inputs, colors={SPECIES_LABELS[s]: c for s, c in SPECIES_COLORS.items()},
           counts=counts_6a, order_seed=ORDER_SEED, panel_mm=PANEL_MM, dpi=UMAP_DPI)
display(SVG(filename=str(files_6a[0])))

## 6B · SAMap UMAP, transferred cell type

Each human nucleus takes the mouse cell type (eFib or dec) carrying most of its cross-species edge weight: the vote
for a type is the sum of the connectivities to mouse nuclei of that type (`C_hs_mm`, the human x mouse block of the
SAMap graph). Every mouse nucleus votes, apoptotic decidual ones included. Ties go to the first type in sorted
order. Nuclei with no mouse neighbor are grey.

In [ ]:
# The stored transfer, and the same vote recomputed from the joint graph's human x mouse block as a check.
graph = sp.csr_matrix(samap.obsp["connectivities"])
is_human, is_mouse = (joint.species == "hs").to_numpy(), (joint.species == "mm").to_numpy()
block = graph[is_human][:, is_mouse].tocsr()
hs_names, mm_names = joint.index[is_human], joint.index[is_mouse]
labels = mouse["cell_type"].astype(str).reindex(mm_names)
assert not labels.isna().any(), "mouse nuclei of the graph without a cell type"

types = np.array(sorted(labels.unique()))
onehot = sp.csr_matrix((np.ones(len(labels)), (np.arange(len(labels)), np.searchsorted(types, labels.to_numpy()))),
                       shape=(len(labels), len(types)))
votes = np.asarray((block @ onehot).todense())
total = votes.sum(axis=1)
mapped = total > 1e-9
top = votes.max(axis=1)
transfer = pd.DataFrame({
    "recomputed_cell_type": pd.Series(np.where(mapped, types[votes.argmax(axis=1)], None), index=hs_names,
                                      dtype="object"),
    "vote_share": np.where(mapped, top / np.where(mapped, total, 1), np.nan),
    "cross_species_weight": total,
    "tie": mapped & ((votes == top[:, None]).sum(axis=1) > 1),
}, index=pd.Index(hs_names, name="cell"))
human_6b = human.join(transfer, how="inner").rename(columns={"transferred_mouse_fibdec": "transferred_cell_type"})
assert len(human_6b) == len(human), "human nuclei of the SAMap object missing from the graph block"
stored, recomputed = human_6b.transferred_cell_type, human_6b.recomputed_cell_type
assert (stored.isna() == recomputed.isna()).all() and (stored[stored.notna()] == recomputed[stored.notna()]).all(), \
    "the vote recomputed from obsp['connectivities'] differs from obs['transferred_mouse_fibdec']"
print("6B: the stored transferred cell type equals the vote recomputed from the graph for every human nucleus")
has_label = human_6b.transferred_cell_type.notna()
unknown = set(human_6b.loc[has_label, "transferred_cell_type"]) - set(CELL_TYPE_COLORS)
assert not unknown, f"transferred cell types without a color: {sorted(unknown)}"

agree = has_label & (human_6b.transferred_cell_type == human_6b.cell_type)
agreement = {"agree": int(agree.sum()), "percent_of_labelled": round(100 * agree.sum() / has_label.sum(), 2),
             "percent_of_all_human": round(100 * agree.sum() / len(human_6b), 2),
             "median_vote_share": round(float(human_6b.vote_share.median()), 4)}
counts_6b = {"human_labelled": int(has_label.sum()), "human_no_mouse_neighbor": int((~has_label).sum()),
             "mouse_context": len(mouse), "exact_ties": int(human_6b.tie.sum()),
             "transferred": {str(k): int(v) for k, v in human_6b.transferred_cell_type.value_counts().items()}}
print("6B:", counts_6b)
print("agreement with the human labels:", agreement)

colored = human_6b[has_label].sample(frac=1, random_state=ORDER_SEED)
files_6b = write_umap_panels(
    "Fig6B_umap_transferred_cell_type", "Cell identity transferred onto human cells",
    colored, colored.transferred_cell_type.map(CELL_TYPE_COLORS).to_list(), human_6b[~has_label],
    f"human, no mouse\nneighbor ({counts_6b['human_no_mouse_neighbor']})",
    key_top_right=[(CELL_TYPE_COLORS[t], CELL_TYPE_LABELS[t]) for t in ("endometrial_fibroblast", "decidual_cell")])
table_6b = OUT / "Fig6B_transferred_cell_type.csv"
human_6b[["cell_type", "transferred_cell_type", "vote_share", "cross_species_weight"]].rename(
    columns={"cell_type": "native_cell_type"}).to_csv(table_6b)
provenance("Fig6B_umap_transferred_cell_type", files_6b + [table_6b], inputs,
           operator="obs['transferred_mouse_fibdec']; checked against the argmax over mouse cell_type of the summed "
                    "SAMap connectivities to mouse neighbors",
           colors={CELL_TYPE_LABELS[t]: c for t, c in CELL_TYPE_COLORS.items()}, counts=counts_6b,
           agreement_with_native=agreement, order_seed=ORDER_SEED, panel_mm=PANEL_MM, dpi=UMAP_DPI)
display(SVG(filename=str(files_6b[0])))

# note, this is sometimes not rendering correctly in a jupyter notebook?

## 6C · SAMap UMAP, transferred pseudotime

Each human nucleus colored by the mouse pseudotime transferred to it (the edge-weighted mean over its mouse
neighbors with a pseudotime). Nuclei without a transferred value are grey: no mouse neighbor, or only apoptotic
decidual neighbors, which carry no pseudotime.

In [ ]:
has_value = human[TRANSFERRED].notna()
counts_6c = {"human_with_value": int(has_value.sum()), "human_no_value": int((~has_value).sum()),
             "mouse_context": len(mouse)}
print("6C:", counts_6c)

colored = human[has_value].sample(frac=1, random_state=ORDER_SEED)
files_6c = write_umap_panels(
    "Fig6C_umap_transferred_pseudotime", "Pseudotime transferred onto human cells",
    colored, CMAP(NORM(colored[TRANSFERRED].to_numpy())), human[~has_value],
    f"no transferred\nvalue ({counts_6c['human_no_value']})")
provenance("Fig6C_umap_transferred_pseudotime", files_6c,
           inputs,
           transferred_column=TRANSFERRED, rescaled_or_reranked=False, colormap="viridis, fixed 0-1",
           counts=counts_6c, order_seed=ORDER_SEED, panel_mm=PANEL_MM, dpi=UMAP_DPI)
display(SVG(filename=str(files_6c[0])))

## 6E · transferred pseudotime on the human sections (menstrual, secretory, key)

Each phase is one physical section whose three 10x lanes share a coordinate frame, drawn as one bare 6 x 6 in panel
on black (300 dpi, no text) plus a horizontal color key. Crops and rotations are those of the figure's first version
(14 Oct 2026). Nuclei without a transferred value are drawn in grey underneath.

In [ ]:
# Per phase: crop (x and y limits in µm, None for the whole section), counter-clockwise rotation (degrees) and
# whether the rotated section is mirrored left-right about its center.
SECTIONS = {
    "secretory": {"limits": [(400, 5500), (750, 5250)], "rotation": -90, "mirror": True},
    "menstrual": {"limits": None, "rotation": 90, "mirror": True},
}
NO_VALUE_SECTION = "#B0B0B0"
SPATIAL_IN = 6     # bare square panel, in inches
SPATIAL_DPI = 300
SPATIAL_DOT = 8    # pt^2 on the 6 in panel


def rotate(x, y, degrees, cx, cy):
    """Rotate points counter-clockwise by `degrees` about (cx, cy)."""
    if degrees % 360 == 0:
        return x, y
    t = np.deg2rad(degrees)
    return (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx, (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy


def section_frame(cells: pd.DataFrame, limits, degrees, mirror: bool) -> pd.DataFrame:
    """Nuclei inside the crop, with coordinates xr, yr (µm) rotated and, if mirror, mirrored left-right."""
    x, y = cells["x_um"].to_numpy(float), cells["y_um"].to_numpy(float)
    if limits is not None:
        (x0, x1), (y0, y1) = limits
        keep = (x >= x0) & (x <= x1) & (y >= y0) & (y <= y1)
        cells, x, y = cells[keep], x[keep], y[keep]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    else:
        cx, cy = (x.min() + x.max()) / 2, (y.min() + y.max()) / 2
    xr, yr = rotate(x, y, degrees, cx, cy)
    if mirror:
        xr = 2 * cx - xr  # left-right, about the center the section was rotated around
    return cells.assign(xr=xr, yr=yr)


def marker_padding(span_x, span_y, dot_size) -> tuple:
    """Pad the data range by one marker radius (+2 px) so edge dots are not clipped on the bare panel."""
    radius_px = np.sqrt(dot_size / np.pi) / 72 * SPATIAL_DPI + 2
    side_px = SPATIAL_IN * SPATIAL_DPI

    def pad(span):
        return span * (radius_px / side_px) / (1 - 2 * radius_px / side_px) + 0.005 * span

    return pad(span_x), pad(span_y)


def spatial_panel(frame: pd.DataFrame, values, path: Path, no_value_color: str) -> list:
    """Bare black square: nuclei without a value underneath in grey, the rest colored by value (fixed 0-1)."""
    fig, ax = plt.subplots(figsize=(SPATIAL_IN, SPATIAL_IN), facecolor="black")
    ax.set_position([0, 0, 1, 1])
    ax.set_facecolor("black")
    pad_x, pad_y = marker_padding(np.ptp(frame.xr), np.ptp(frame.yr), SPATIAL_DOT)
    ax.set_xlim(frame.xr.min() - pad_x, frame.xr.max() + pad_x)
    ax.set_ylim(frame.yr.min() - pad_y, frame.yr.max() + pad_y)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    missing = ~np.isfinite(values)
    ax.scatter(frame.xr[missing], frame.yr[missing], s=SPATIAL_DOT, color=no_value_color, linewidths=0, zorder=1)
    ax.scatter(frame.xr[~missing], frame.yr[~missing], s=SPATIAL_DOT, c=CMAP(NORM(values[~missing])), linewidths=0,
               zorder=2)
    written = [path.with_suffix(".svg")]
    fig.savefig(written[0], dpi=SPATIAL_DPI, facecolor="black")
    plt.close(fig)
    return written


def spatial_key(path: Path, title: str) -> list:
    """Horizontal color key for the section panels."""
    fig = plt.figure(figsize=(3.0, 1.0), facecolor="black")
    cax = fig.add_axes([0.12, 0.45, 0.76, 0.2])
    bar = fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=cax, orientation="horizontal",
                       ticks=[0, 0.5, 1])
    bar.ax.set_xticklabels(["0 early", "0.5", "1 late"], color=TEXT, fontsize=8)
    bar.outline.set_visible(False)
    bar.ax.tick_params(color=TEXT, length=2)
    cax.set_title(title, color=TEXT, fontsize=8, pad=4)
    written = [path.with_suffix(".svg")]
    fig.savefig(written[0], dpi=300, facecolor="black")
    plt.close(fig)
    return written



def type_panel(frame: pd.DataFrame, labelled, colors, path: Path) -> list:
    """Bare black square: nuclei with no mouse neighbor underneath in grey, the rest by transferred cell type."""
    fig, ax = plt.subplots(figsize=(SPATIAL_IN, SPATIAL_IN), facecolor="black")
    ax.set_position([0, 0, 1, 1])
    ax.set_facecolor("black")
    pad_x, pad_y = marker_padding(np.ptp(frame.xr), np.ptp(frame.yr), SPATIAL_DOT)
    ax.set_xlim(frame.xr.min() - pad_x, frame.xr.max() + pad_x)
    ax.set_ylim(frame.yr.min() - pad_y, frame.yr.max() + pad_y)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    ax.scatter(frame.xr[~labelled], frame.yr[~labelled], s=SPATIAL_DOT, color=NO_VALUE_SECTION, linewidths=0, zorder=1)
    # Both cell types go down in one shuffled layer, so neither hides the other where they overlap.
    order = np.random.default_rng(ORDER_SEED).permutation(int(labelled.sum()))
    ax.scatter(frame.xr[labelled].to_numpy()[order], frame.yr[labelled].to_numpy()[order], s=SPATIAL_DOT,
               c=colors[labelled][order], linewidths=0, zorder=2)
    written = [path.with_suffix(".svg")]
    fig.savefig(written[0], dpi=SPATIAL_DPI, facecolor="black")
    plt.close(fig)
    return written


def type_key(path: Path) -> list:
    """Dot key for the cell-type section panels."""
    fig = plt.figure(figsize=(3.0, 1.0), facecolor="black")
    entries = [(CELL_TYPE_COLORS[t], CELL_TYPE_LABELS[t]) for t in ("endometrial_fibroblast", "decidual_cell")]
    entries.append((NO_VALUE_SECTION, "no mouse neighbor"))
    handles = [Line2D([], [], ls="", marker="o", ms=6, mfc=c, mec="none", label=l) for c, l in entries]
    fig.legend(handles=handles, loc="center", ncol=3, fontsize=8, labelcolor=TEXT, frameon=False,
               title="transferred from mouse", title_fontsize=8, handletextpad=0.3, columnspacing=1.2)
    fig.legends[0].get_title().set_color(TEXT)
    written = [path.with_suffix(".svg")]
    fig.savefig(written[0], dpi=300, facecolor="black")
    plt.close(fig)
    return written

files_6e, counts_6e = [], {}
for phase, spec in SECTIONS.items():
    cells = human[(human.timepoint.astype(str) == phase) & human.x_um.notna()]
    frame = section_frame(cells, spec["limits"], spec["rotation"], spec["mirror"])
    files_6e += spatial_panel(frame, frame[TRANSFERRED].to_numpy(),
                              OUT / f"Fig6E_{DATE}-Human-{phase}-transferred_pseudotime-spatial.svg", NO_VALUE_SECTION)
    counts_6e[phase] = {"libraries": sorted(frame["sample"].astype(str).unique()), "drawn": len(frame),
                        "no_value": int(frame[TRANSFERRED].isna().sum()),
                        "median_transferred": round(float(frame[TRANSFERRED].median()), 4), **spec}
    print(f"6E {phase}: {counts_6e[phase]}")
files_6e += spatial_key(OUT / f"Fig6E_{DATE}-spatial_key_transferred_pseudotime.svg", "transferred pseudotime (rank)")
provenance("Fig6E_sections_transferred_pseudotime", files_6e,
           inputs,
           transferred_column=TRANSFERRED, rescaled_or_reranked=False, colormap="viridis, fixed 0-1",
           sections=counts_6e, panel_in=SPATIAL_IN, dpi=SPATIAL_DPI)
for path in files_6e:
    display(SVG(filename=str(path)))

## 6D · transferred cell type on the human sections (menstrual, secretory)

The 6B transfer on the tissue: each positioned human nucleus colored by the mouse cell type carrying most of its
cross-species edge weight (eFib, dec; the 6B colors), nuclei with no mouse neighbor in grey underneath. Same crops,
rotations and panel size as 6E.

In [ ]:
files_6d, counts_6d = [], {}
for phase, spec in SECTIONS.items():
    cells = human_6b[(human_6b.timepoint.astype(str) == phase) & human_6b.x_um.notna()]
    frame = section_frame(cells, spec["limits"], spec["rotation"], spec["mirror"])
    labelled = frame.transferred_cell_type.notna().to_numpy()
    colors = frame.transferred_cell_type.map(CELL_TYPE_COLORS).to_numpy()
    files_6d += type_panel(frame, labelled, colors,
                           OUT / f"Fig6d_{DATE}-Human-{phase}-transferred_cell_type-spatial.svg")
    counts_6d[phase] = {"drawn": len(frame), "no_mouse_neighbor": int((~labelled).sum()),
                        "transferred": {str(k): int(v) for k, v in frame.transferred_cell_type.value_counts().items()},
                        **spec}
    print(f"6d {phase}: {counts_6d[phase]}")
files_6d += type_key(OUT / f"Fig6d_{DATE}-spatial_key_transferred_cell_type.svg")
provenance("Fig6D_sections_transferred_cell_type", files_6d,
           inputs,
           operator="as 6B", colors={CELL_TYPE_LABELS[t]: c for t, c in CELL_TYPE_COLORS.items()},
           sections=counts_6d, panel_in=SPATIAL_IN, dpi=SPATIAL_DPI)
for path in files_6d:
    display(SVG(filename=str(path)))

In [ ]:
# 6D: mouse cell type on the 1.5 dpi late and 4.5 dpi menstrual sections, in the 6B colors, framed as
# Ç. Cevrim's cell-type panels (whole section rotated about its center, then a square view window;
# 6 x 6 in, 300 dpi, 25 pt² dots). One SVG per section plus a key.
# Uses names from the cells above: mouse, inputs, CELL_TYPE_COLORS, CELL_TYPE_LABELS, rotate, ORDER_SEED,
# SPATIAL_IN, SPATIAL_DPI, TEXT, provenance, file_record, OUT, DATE.
MOUSE_H5AD = Path("/n/lab_storage/eddy_lab/Lab/mckinley/data/20261110_mouse_only_eFib_dec_final_nick_pdt.h5ad")

# Per timepoint: libraries, rotation (degrees, counter-clockwise) and the square view window (µm, rotated frame).
MOUSE_SECTIONS = {
    "1.5dpi_late": {"samples": ["1.5dpi_late_mck12a", "1.5dpi_late_mck12b"], "rotation": 303,
                    "view": [(150, 6250), (-300, 5800)]},
    "4.5dpi_menstrual": {"samples": ["4.5dpi_menstrual_mck11a"], "rotation": 81,
                         "view": [(900, 9700), (1350, 10150)]},
}
MOUSE_DOT = 25         # pt^2 on the 6 in panel, as the cell-type panels
SHORT_TYPE_COLORS = {CELL_TYPE_LABELS[t]: c for t, c in CELL_TYPE_COLORS.items()}  # eFib, dec -> 6B colors

# The mouse object: positions, timepoint and cell type (eFib / dec) of every mouse nucleus of the SAMap object.
mouse_obs = ad.read_h5ad(MOUSE_H5AD, backed="r").obs[["sample", "timepoint", "cell_type", "x_um_dbscan",
                                                       "y_um_dbscan"]]
mouse_obs = mouse_obs.astype({"sample": str, "timepoint": str, "cell_type": str})
mouse_obs.index.name = "cell"
assert set(mouse.index) == set(mouse_obs.index), "mouse nuclei of the SAMap object differ from the mouse object"
same_label = mouse["cell_type"].map(CELL_TYPE_LABELS).reindex(mouse_obs.index) == mouse_obs["cell_type"]
assert same_label.all(), "cell type in the mouse object differs from the SAMap object's"
inputs_6d = inputs + [file_record(MOUSE_H5AD, "mouse_object")]


def mouse_section_frame(cells: pd.DataFrame, degrees) -> pd.DataFrame:
    """Display coordinates xr, yr (µm): the whole section rotated about its center."""
    x, y = cells["x_um_dbscan"].to_numpy(float), cells["y_um_dbscan"].to_numpy(float)
    cx, cy = (x.min() + x.max()) / 2, (y.min() + y.max()) / 2
    xr, yr = rotate(x, y, degrees, cx, cy)
    return cells.assign(xr=xr, yr=yr)


def draw_mouse_section(ax, frame: pd.DataFrame, view, dot: float) -> None:
    """Nuclei by cell type inside the view window; both types in one shuffled layer, as 6F."""
    (x0, x1), (y0, y1) = view
    ax.set_facecolor("black")
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    order = np.random.default_rng(ORDER_SEED).permutation(len(frame))
    ax.scatter(frame.xr.to_numpy()[order], frame.yr.to_numpy()[order], s=dot,
               c=frame["cell_type"].map(SHORT_TYPE_COLORS).to_numpy()[order], linewidths=0)


files_6d, counts_6d = [], {}
for timepoint, spec in MOUSE_SECTIONS.items():
    cells = mouse_obs[mouse_obs["sample"].isin(spec["samples"]) & mouse_obs.x_um_dbscan.notna()]
    frame = mouse_section_frame(cells, spec["rotation"])
    fig, ax = plt.subplots(figsize=(SPATIAL_IN, SPATIAL_IN), facecolor="black")
    ax.set_position([0, 0, 1, 1])
    draw_mouse_section(ax, frame, spec["view"], MOUSE_DOT)
    path = OUT / f"Fig6D_{DATE}-Mouse-{timepoint}-cell_type-spatial.svg"
    fig.savefig(path, dpi=SPATIAL_DPI, facecolor="black")
    plt.close(fig)
    files_6d.append(path)
    (x0, x1), (y0, y1) = spec["view"]
    in_view = frame[(frame.xr >= x0) & (frame.xr <= x1) & (frame.yr >= y0) & (frame.yr <= y1)]
    counts_6d[timepoint] = {"positioned": len(frame), "in_view": len(in_view),
                            "cell_types_in_view": {k: int(v) for k, v in in_view["cell_type"].value_counts().items()},
                            **spec}
    print(f"6D {timepoint}: {counts_6d[timepoint]}")

# Key: the two cell types only (every mouse nucleus has a label).
fig = plt.figure(figsize=(3.0, 1.0), facecolor="black")
handles = [Line2D([], [], ls="", marker="o", ms=6, mfc=c, mec="none", label=l) for l, c in SHORT_TYPE_COLORS.items()]
fig.legend(handles=handles, loc="center", ncol=2, fontsize=8, labelcolor=TEXT, frameon=False,
           title="mouse cell type", title_fontsize=8, handletextpad=0.3, columnspacing=1.2)
fig.legends[0].get_title().set_color(TEXT)
key_path = OUT / f"Fig6D_{DATE}-spatial_key_cell_type.svg"
fig.savefig(key_path, dpi=300, facecolor="black")
plt.close(fig)
files_6d.append(key_path)
provenance("Fig6D_mouse_sections_cell_type", files_6d, inputs_6d,
           colors=SHORT_TYPE_COLORS, positions="x_um_dbscan, y_um_dbscan of the mouse object (no spatial QC)",
           framing="whole section rotated about its center, then the view window (Ç. Cevrim, 11 Nov 2026)",
           sections=counts_6d, panel_in=SPATIAL_IN, dpi=SPATIAL_DPI, dot_pt2=MOUSE_DOT)
for path in files_6d:
    display(SVG(filename=str(path)))


In [ ]:
print("wrote", sorted(p.name for p in OUT.iterdir()))